In [3]:

import os
import psycopg
from dotenv import load_dotenv
from pgvector.psycopg import register_vector
from sentence_transformers import SentenceTransformer

load_dotenv("../.env")
conn = psycopg.connect(
    host=os.getenv("DB_HOST"), port=os.getenv("DB_PORT"),
    dbname=os.getenv("DB_NAME"), user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
)
register_vector(conn)

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Ready")

def search(query, k=5, year_from=None, year_to=None):
    q_vec = model.encode(query, normalize_embeddings=True)

    where, params = [], [q_vec]
    if year_from is not None:
        where.append("year >= %s"); params.append(year_from)
    if year_to is not None:
        where.append("year <= %s"); params.append(year_to)
    where_sql = ("WHERE " + " AND ".join(where)) if where else ""

    sql = f"""
        SELECT DISTINCT ON (song_id) song_id, title, artist, year,
               1 - (embedding <=> %s) AS similarity
        FROM chunks {where_sql}
        ORDER BY song_id, embedding <=> %s
    """
    rows = conn.execute(sql, params[:1] + params[1:] + [q_vec]).fetchall()
    rows.sort(key=lambda r: r[4], reverse=True)
    return rows[:k]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready


In [4]:
for r in search("feeling lonely after a breakup"): print(r)
print()
for r in search("feeling lonely after a breakup", year_from=1990, year_to=1999): print(r)
print()
for r in search("dancing all night at a party", year_from=1970, year_to=1979): print(r)

(2304, 'Even the Nights Are Better', 'air supply', 1982, 0.6459550857543945)
(3846, 'Back 2 Good', 'matchbox twenty', 1999, 0.5987488031387329)
(4079, 'U Got It Bad', 'usher', 2002, 0.5602535009384155)
(2141, "We Don't Talk Anymore - 2001 Remaster", 'cliff richard', 1980, 0.5568995641738049)
(3261, 'We Got A Love Thang', 'cece peniston', 1992, 0.5525043771417256)

(3846, 'Back 2 Good', 'matchbox twenty', 1999, 0.5987488031387329)
(3261, 'We Got A Love Thang', 'cece peniston', 1992, 0.5525043771417256)
(3510, 'Brokenhearted (feat. Wanya Morris) - Single Version', 'brandy, wanya morris', 1995, 0.5252357721328735)
(3874, 'Faded Pictures', 'case, joe', 1999, 0.4815535258320861)
(3392, 'Anytime You Need a Friend', 'mariah carey', 1994, 0.4808104049676424)

(1859, 'Dancing Queen', 'abba', 1977, 0.5447430610656777)
(1925, 'Boogie Nights', 'heatwave', 1977, 0.5056436651177806)
(2098, 'Dance the Night Away - 2015 Remaster', 'van halen', 1979, 0.48595389807499156)
(1789, 'You Should Be Dancing',